# Great Britain race-population completeness audit

## Bounded audit question

> Are any Great Britain races that officially produced results missing from Source Version 1 / Database v4?

This is a **database/source-correctness audit**, not a reader-facing study. It follows directly from Great Britain Study 04.

The decisive defect test is:

> **official BHA completed race result -> corresponding Inside Rails source race occurrence**

Scheduled fixture evidence is also useful, but for a different purpose: it helps explain races or fixtures that were abandoned, cancelled, transferred, rescheduled or otherwise changed. A scheduled race or fixture that does not appear in Source Version 1 is **not by itself** evidence of a source-population defect.


## Audit controls and inherited governance

Read before this audit:

- `docs/STUDY_DATABASE_REFERENCE.md`
- `docs/STUDY_DATA_ACCESS.md`
- `docs/RESEARCH_DATA_SOURCE_REGISTER.md`
- `docs/studies/GB_04_RACE_MEETINGS_AND_FIXTURES_CLOSEOUT.md`
- `docs/STUDY_CLOSEOUT_REGISTER.md`

Inherited boundaries:

- Source Version 1 remains immutable and read-only.
- Source admission remains `rowid <> 1`.
- Authorised Source Version 1 race identity remains exact raw `date + course + off`.
- Database v4 is the accepted immutable analytical release.
- Do **not** create a fixture ID, meeting ID or session ID for this audit.
- Do **not** assume `date + racecourse = fixture`.
- Database v4's governed British racecourse identity is the correct geographical bridge for comparison, but it does not identify a physical course/track below racecourse level.


In [ ]:
# Audit setup
#
# Establish the exact accepted database and the bounded Great Britain source
# population before acquiring or reconciling any external BHA evidence.

from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")
DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v4.sqlite3"
)

assert DATABASE.is_file(), f"Accepted Database v4 not found: {DATABASE}"

with connect_read_only(DATABASE) as connection:
    database_version = connection.execute("PRAGMA user_version").fetchone()[0]
    population_summary = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS gb_race_occurrences,
            COUNT(DISTINCT source_race_occurrence_code) AS distinct_race_codes,
            MIN(raw_date) AS minimum_source_date,
            MAX(raw_date) AS maximum_source_date,
            COUNT(DISTINCT raw_date) AS source_dates,
            COUNT(DISTINCT candidate_course_label) AS source_course_labels,
            COUNT(DISTINCT racecourse_identity_code) AS governed_racecourses
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        """,
        connection,
    )

assert database_version == 4, f"Expected Database v4, found user_version={database_version}"
assert int(population_summary.loc[0, "gb_race_occurrences"]) == 111_634
assert int(population_summary.loc[0, "distinct_race_codes"]) == 111_634

print(f"Database v{database_version} confirmed.")
population_summary


## Reconciliation model

The audit uses three evidence layers without pretending they are the same thing:

1. **BHA scheduled racing** — what was intended or programmed;
2. **BHA official results** — what actually produced a result;
3. **Source Version 1 / Database v4** — what the third-party source population contains.

The principal completeness denominator is layer 2, not layer 1.

Schedule -> results reconciliation is explanatory. Results -> Database v4 reconciliation is the correctness test.

### Initial classification vocabulary

Keep classifications descriptive until evidence supports something stronger:

- `official_result_matched_source`
- `official_result_candidate_source_match_requires_review`
- `official_result_unmatched_after_reconciliation`
- `scheduled_no_official_result`
- `schedule_changed_or_transferred`
- `external_evidence_unresolved`

Only a fully investigated `official_result_unmatched_after_reconciliation` case can become a genuine `official_result_missing_from_source` defect.


## Authoritative external evidence

Primary source: British Horseracing Authority.

### Results service

`https://www.britishhorseracing.com/racing/results/`

The public results interface exposes realised fixture characteristics including fixture date, fixture type, racecourse, first race and fixture session, and distinguishes fixtures marked **Abandoned** from those for which results can be viewed.

### Fixture lists

`https://www.britishhorseracing.com/racing/fixtures/full-year/`

BHA publishes annual fixture lists. Current material is downloadable, including Excel/PDF formats. The BHA page states that older fixture details can be requested from the Racing Department.

### Evidence boundary

Do not infer a persistent BHA fixture identifier from displayed date/course/session fields. Study 04 already established that the public results description is not a durable fixture key.


## Proposed audit evidence grains

These are **audit artifacts**, not database schema proposals.

### Official result race evidence

One row per BHA race for which the official results service provides a result. Preserve, where exposed:

- result date;
- BHA racecourse name;
- fixture/session context;
- race time / race ordering information;
- race name or description;
- stable public result URL or other BHA reference if available;
- enough result detail to disambiguate a candidate match where necessary;
- retrieval timestamp and source locator.

### Scheduled fixture evidence

One row per published BHA scheduled fixture observation, preserving the fields actually supplied by the relevant annual list. Do not invent missing race-programme detail.

The annual fixture list can support schedule/result context even if it is not sufficient to reconstruct every historical scheduled race.


## Matching strategy

Do not match using raw source `race_id`.

Candidate matching should start with the strongest common observable facts available on both sides, normally:

- official/result date;
- governed racecourse identity after explicit BHA-name -> Inside Rails racecourse reconciliation;
- race time / source raw `off` where semantically comparable.

If that is not unique or a time has changed, use additional race evidence such as race name and runner/result signatures to resolve the case.

A failed first-pass join is an **investigation queue**, not proof of a missing race. Transfers, racecourse naming differences, time amendments and programme changes must be exhausted first.


In [ ]:
# Database-side pilot population
#
# Use the final Source Version 1 date as a small concrete slice while we
# establish the reproducible BHA result-acquisition route. This does not yet
# attempt any external match.

pilot_date = "2026-05-27"

with connect_read_only(DATABASE) as connection:
    db_pilot = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date,
            raw_course,
            raw_off,
            candidate_course_label,
            governed_racecourse_name,
            racecourse_identity_code,
            race_name_raw,
            governed_runner_count
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        ORDER BY governed_racecourse_name, raw_off, source_race_occurrence_code
        """,
        connection,
        params=(pilot_date,),
    )

print(f"Database v4 races on {pilot_date}: {len(db_pilot):,}")
db_pilot


## Question 1 — Can the BHA completed-result population be acquired reproducibly?

### Why this comes first

A source-completeness percentage is meaningless unless the external denominator itself is demonstrably complete for the requested period.

### Smallest next test

Before writing a bulk collector:

1. inspect the BHA results interface and its underlying request/response structure;
2. retrieve one bounded date with known realised racing;
3. prove that every fixture and individual race displayed for that date can be represented without relying on an inferred fixture identity;
4. compare that single-date official result population with `db_pilot`;
5. only then generalise the acquisition method across the Source Version 1 period.

Do not bulk-scrape first and discover later that pagination, abandoned fixtures or historical limits were misunderstood.


## 2026 bounded acquisition case

The first generalisation step is now deliberately **the whole of 2026**, not the whole 2015-2026 source period.

This gives the audit a live-year case in which the original published plan can be compared with later fixture state and realised official results before the acquisition method is applied historically.

### Keep three 2026 evidence layers separate

1. **Original published 2026 fixture plan** — the BHA annual fixture list as originally published.
2. **Observed 2026 fixture state** — later BHA observations showing what is scheduled, altered, transferred, cancelled or abandoned at retrieval time.
3. **Official 2026 race results** — individual races that actually produced official results.

The original annual plan must not be overwritten by later observations. Fixture evidence is mutable; each observation requires provenance and retrieval time.

### Source-completeness boundary

The actual Source Version 1 / Database v4 completeness test remains bounded to **2026-01-01 through 2026-05-27**, because Source Version 1 ends on 2026-05-27.

Official results after 2026-05-27 are useful for validating the BHA acquisition method, but they are outside Source Version 1 coverage and cannot be classified as source omissions. Future scheduled fixtures are planning observations only and must never enter the completed-race denominator.

### Preserve the raw BHA documents

Static BHA 2026 evidence is retained unchanged under `data/external/bha/gb_race_population_completeness/2026/`. The bounded downloader `scripts/download_bha_2026_audit_evidence.py` records source URL, retrieval timestamp, byte size and SHA-256 in `manifest.json`. The retained set includes the 2026 Fixture List PDF/Excel, Headline Measures PDF and January-May 2026 Racing Data Packs.

### Revised progression

1. Prove the results acquisition route on **2026-05-27**.
2. Generalise the same route to all of **2026**.
3. Reconcile original plan -> observed fixture state -> official results without treating those layers as interchangeable.
4. Compare official results through 2026-05-27 with Database v4.
5. Only after the 2026 method is understood and reproducible, extend historical result acquisition across 2015-2025.


### Step 1 — Inspect the original 2026 BHA fixture workbook

Before parsing the annual fixture list, inspect the retained original **inside this notebook**.

The purpose of this step is deliberately narrow:

- confirm the workbook and worksheets we actually received from BHA;
- inspect the row/column structure of `List - Full Year`;
- identify the supplied header fields and the apparent row grain;
- **do not yet assume** that one spreadsheet row is a governed fixture identity;
- **do not modify** the retained XLSX.

The workbook is planning evidence. Even if its list sheet proves to be one row per published fixture observation, it remains the **original plan**, not the completed-race denominator.


In [ ]:
# Inspect the retained BHA 2026 fixture workbook in-place and read-only.
#
# We intentionally use only Python's standard library here. An XLSX file is a
# ZIP archive of XML documents, so this lets the audit inspect the source
# without introducing a new project dependency merely for this first look.
#
# This cell is exploratory evidence inspection, not a parser. We are looking
# for the workbook's actual structure and displayed values before deciding
# what a governed extraction should look like.

import re
import zipfile
import xml.etree.ElementTree as ET

BHA_2026_EVIDENCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "bha"
    / "gb_race_population_completeness"
    / "2026"
)
FIXTURE_WORKBOOK = BHA_2026_EVIDENCE_DIR / "2026_Fixture_List.xlsx"

assert FIXTURE_WORKBOOK.is_file(), f"BHA fixture workbook not found: {FIXTURE_WORKBOOK}"

# XLSX namespace constants. Keeping them explicit makes the XML traversal
# auditable rather than hiding workbook interpretation behind a convenience
# library.
MAIN_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
REL_NS = "http://schemas.openxmlformats.org/package/2006/relationships"
DOC_REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
NS = {"x": MAIN_NS}
RELATIONSHIP_NS = {"r": REL_NS}

with zipfile.ZipFile(FIXTURE_WORKBOOK) as archive:
    workbook_xml = ET.fromstring(archive.read("xl/workbook.xml"))
    workbook_relationships = ET.fromstring(
        archive.read("xl/_rels/workbook.xml.rels")
    )

    # Map workbook relationship IDs to the worksheet XML files they point to.
    relationship_targets = {
        relationship.attrib["Id"]: relationship.attrib["Target"]
        for relationship in workbook_relationships.findall(
            "r:Relationship", RELATIONSHIP_NS
        )
    }

    sheet_targets = {}
    for sheet in workbook_xml.find("x:sheets", NS):
        relationship_id = sheet.attrib[f"{{{DOC_REL_NS}}}id"]
        target = relationship_targets[relationship_id]
        sheet_targets[sheet.attrib["name"]] = f"xl/{target}"

    print("Workbook sheets:")
    for name, target in sheet_targets.items():
        print(f"  - {name}: {target}")

    # The workbook exposes several presentation/summary sheets. We inspect
    # 'List - Full Year' because its name suggests the row-level annual list,
    # but we still test its contents rather than assuming its grain.
    LIST_SHEET_NAME = "List - Full Year"
    assert LIST_SHEET_NAME in sheet_targets, (
        f"Expected worksheet {LIST_SHEET_NAME!r}; found {list(sheet_targets)}"
    )

    # Shared strings are stored separately in many XLSX workbooks. Build the
    # lookup only if the workbook actually contains that part.
    shared_strings = []
    if "xl/sharedStrings.xml" in archive.namelist():
        shared_strings_xml = ET.fromstring(archive.read("xl/sharedStrings.xml"))
        for item in shared_strings_xml.findall("x:si", NS):
            shared_strings.append(
                "".join(text.text or "" for text in item.iter(f"{{{MAIN_NS}}}t"))
            )

    sheet_xml = ET.fromstring(archive.read(sheet_targets[LIST_SHEET_NAME]))
    dimension = sheet_xml.find("x:dimension", NS)
    print(
        "\nDeclared used range:",
        dimension.attrib.get("ref") if dimension is not None else None,
    )

    def cell_value(cell):
        # Return the stored/displayed value for a worksheet cell.
        cell_type = cell.attrib.get("t")
        value_node = cell.find("x:v", NS)

        if cell_type == "inlineStr":
            inline = cell.find("x:is", NS)
            if inline is None:
                return None
            return "".join(
                text.text or "" for text in inline.iter(f"{{{MAIN_NS}}}t")
            )

        if value_node is None:
            return None

        value = value_node.text
        if cell_type == "s":
            return shared_strings[int(value)]
        if cell_type == "b":
            return value == "1"
        return value

    def column_number(cell_reference):
        # Convert an Excel reference such as C12 to its 1-based column number.
        letters = re.match(r"[A-Z]+", cell_reference).group(0)
        number = 0
        for letter in letters:
            number = number * 26 + (ord(letter) - ord("A") + 1)
        return number

    # Preserve blank cells between populated cells so the preview reflects the
    # actual worksheet column positions rather than collapsing them.
    preview_rows = []
    sheet_data = sheet_xml.find("x:sheetData", NS)
    for row in list(sheet_data)[:15]:
        values_by_column = {
            column_number(cell.attrib["r"]): cell_value(cell)
            for cell in row.findall("x:c", NS)
        }
        last_column = max(values_by_column, default=0)
        preview_rows.append(
            [values_by_column.get(column) for column in range(1, last_column + 1)]
        )

# Pad rows to a common width only for display. No interpretation or cleaning
# has happened yet.
preview_width = max(map(len, preview_rows), default=0)
fixture_workbook_preview = pd.DataFrame(
    [row + [None] * (preview_width - len(row)) for row in preview_rows]
)

print("\nFirst 15 stored rows from 'List - Full Year':")
fixture_workbook_preview


#### Interpretation checkpoint

Stop here after running the cell.

The output should tell us what BHA actually supplied: worksheet names, the declared used range, the header structure and the first fixture rows. **Only then** should the next cell turn the sheet into a structured fixture-plan table.

In particular, we should not write a parser that silently treats a row as a fixture until the displayed workbook structure supports that interpretation.
